# Transformation 01 — Type Casting & Ordinal Encoding

1. **Risk → ordinal**: `Low` -> 1, `Medium` -> 2, `High` -> 3
2. **Date columns → datetime**: Convert `LICENSE TERM START DATE`, `LICENSE TERM EXPIRATION DATE`,
   and `Inspection Date` to datetime; extract temporal features.
3. **License duration**: Compute `license_duration_days` = expiration − start.
4. **Handle missing dates**: FLAG missing `LICENSE TERM START DATE` values;
   impute with median where needed.

In [12]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.transformed('train_t00.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t00.parquet'))
print('Train shape:', train.shape)
print('Test  shape:', test.shape)
print()
print('Train columns:', list(train.columns))

Train shape: (137176, 26)
Test  shape: (34294, 26)

Train columns: ['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'APPLICATION TYPE', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE', 'LICENSE STATUS', 'flag_non_il_state', 'flag_non_chicago_city', 'flag_longitude_outside_typical_range', 'violations_recorded', 'license_matched', 'has_prior_inspection']


## 1 · Risk — Ordinal Encoding

Map the `Risk` column to numeric ordinal values:
- `High` -> 3
- `Medium` -> 2
- `Low` -> 1

In [13]:
print('=== Risk unique values (before) ===')
print(train['Risk'].value_counts())
print()

RISK_MAP = {
    'High': 3,
    'Medium': 2,
    'Low': 1,
    'Unknown': 3 # mode imputation for missing values
}

unexpected_train = set(train['Risk'].dropna().unique()) - set(RISK_MAP.keys())
unexpected_test  = set(test['Risk'].dropna().unique()) - set(RISK_MAP.keys())
if unexpected_train:
    print(f'WARNING: unexpected Risk values in train: {unexpected_train}')
if unexpected_test:
    print(f'WARNING: unexpected Risk values in test: {unexpected_test}')

train['Risk'] = train['Risk'].map(RISK_MAP)
test['Risk']  = test['Risk'].map(RISK_MAP)

print('=== Risk unique values (after) ===')
print(train['Risk'].value_counts())

=== Risk unique values (before) ===
Risk
High       99437
Medium     27549
Low        10161
Unknown       29
Name: count, dtype: int64

=== Risk unique values (after) ===
Risk
3    99466
2    27549
1    10161
Name: count, dtype: int64


## 2 · Date columns — convert to datetime & extract features

In [9]:
DATE_COLS = ['Inspection Date', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE']

for col in DATE_COLS:
    if col in train.columns:
        train[col] = pd.to_datetime(train[col], errors='coerce')
        test[col]  = pd.to_datetime(test[col], errors='coerce')
        print(f'{col}: dtype={train[col].dtype}, nulls_train={train[col].isna().sum()}, nulls_test={test[col].isna().sum()}')

Inspection Date: dtype=datetime64[ns], nulls_train=0, nulls_test=0
LICENSE TERM START DATE: dtype=datetime64[ns], nulls_train=117487, nulls_test=29696
LICENSE TERM EXPIRATION DATE: dtype=datetime64[ns], nulls_train=6781, nulls_test=1745


### 2a · Inspection Date features

In [10]:
for df in [train, test]:
    df['inspection_year']       = df['Inspection Date'].dt.year
    df['inspection_month']      = df['Inspection Date'].dt.month
    df['inspection_dayofweek']  = df['Inspection Date'].dt.dayofweek
    df['inspection_quarter']    = df['Inspection Date'].dt.quarter

print('Inspection Date features extracted.')
print(train[['inspection_year', 'inspection_month', 'inspection_dayofweek', 'inspection_quarter']].describe())

Inspection Date features extracted.
       inspection_year  inspection_month  inspection_dayofweek  \
count    137176.000000     137176.000000         137176.000000   
mean       2013.450028          6.337333              2.065186   
std           2.240235          3.308091              1.385338   
min        2010.000000          1.000000              0.000000   
25%        2011.000000          4.000000              1.000000   
50%        2014.000000          6.000000              2.000000   
75%        2015.000000          9.000000              3.000000   
max        2017.000000         12.000000              6.000000   

       inspection_quarter  
count       137176.000000  
mean             2.443037  
std              1.084553  
min              1.000000  
25%              2.000000  
50%              2.000000  
75%              3.000000  
max              4.000000  


### 2b · License duration feature

Compute `license_duration_days` = `LICENSE TERM EXPIRATION DATE` − `LICENSE TERM START DATE`.
Also flag rows where `LICENSE TERM START DATE` was missing.

In [15]:
print('=== LICENSE TERM EXPIRATION DATE nulls ===')
print(f'Train: {train["LICENSE TERM EXPIRATION DATE"].isna().sum()}')
print(f'Test:  {test["LICENSE TERM EXPIRATION DATE"].isna().sum()}')

print()
print('=== Inspection Date nulls ===')
print(f'Train: {train["Inspection Date"].isna().sum()}')
print(f'Test:  {test["Inspection Date"].isna().sum()}')

=== LICENSE TERM EXPIRATION DATE nulls ===
Train: 6781
Test:  1745

=== Inspection Date nulls ===
Train: 0
Test:  0


In [17]:
# days_to_license_expiry: negative = license was expired at inspection time
for df in [train, test]:
    df['days_to_license_expiry'] = (
        pd.to_datetime(df['LICENSE TERM EXPIRATION DATE']) - pd.to_datetime(df['Inspection Date'])
    ).dt.days

# Flag missing before filling
for df in [train, test]:
    df['license_expiry_missing'] = df['days_to_license_expiry'].isna().astype(int)

# Fit median on train only, apply to both
median_expiry = train['days_to_license_expiry'].median()
print(f'Median days to license expiry (train): {median_expiry:.0f} days')

train['days_to_license_expiry'] = train['days_to_license_expiry'].fillna(median_expiry)
test['days_to_license_expiry']  = test['days_to_license_expiry'].fillna(median_expiry)

print(f'Remaining nulls — train: {train["days_to_license_expiry"].isna().sum()}, test: {test["days_to_license_expiry"].isna().sum()}')

Median days to license expiry (train): 2440 days
Remaining nulls — train: 0, test: 0


## 3 · Save intermediate results

In [18]:
train.to_parquet(DataLoader.transformed('train_t01.parquet'), index=False)
test.to_parquet(DataLoader.transformed('test_t01.parquet'), index=False)
print('Saved train_t01.parquet and test_t01.parquet')
print('Train shape:', train.shape)
print('Test  shape:', test.shape)

Saved train_t01.parquet and test_t01.parquet
Train shape: (137176, 28)
Test  shape: (34294, 28)
